# Phase 3.5: Prompt & Data Refinement

**Goal:** Determine if adapter_v3 + best_prompt outperforms base + best_prompt

**Hard Gate:** adapter_v3 + best_prompt > base + best_prompt (delta > 0)

In [ ]:
# 1. Repo/bootstrap setup
# Works in two modes:
# 1) preferred: clone/pull a repo that already contains Phase 3.5 files
# 2) fallback: overlay /content/phase3_5_bundle.zip or /content/drive/MyDrive/codepause_phase3_5/phase3_5_bundle.zip
import os
import pathlib
import shutil
import subprocess
import zipfile

REPO_URL = os.environ.get("CODEPAUSE_REPO_URL", "https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git")
PROJECT_DIR = pathlib.Path(os.environ.get("CODEPAUSE_PROJECT_DIR", "/content/AMDh"))
LOCAL_ROOT = pathlib.Path("/content/codepause_phase3_5")
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/codepause_phase3_5")
root_path = DRIVE_ROOT if DRIVE_ROOT.parent.exists() else LOCAL_ROOT
root_path.mkdir(parents=True, exist_ok=True)
print(f"Artifact copy root: {root_path}")

if not (PROJECT_DIR / ".git").exists():
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout. Set CODEPAUSE_PROJECT_DIR or remove it.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
print(f"Working directory: {pathlib.Path.cwd()}")

required_paths = [
    "src", "eval", "training", "tests",
    "data/thinkanywhere_sft_v3.jsonl",
    "tests/test_phase3_prompt_template.py",
    "tests/test_phase3_evaluator_prompt_support.py",
    "tests/test_phase3_dataset_v3.py",
]

def missing_required():
    return [p for p in required_paths if not pathlib.Path(p).exists()]

missing = missing_required()
if missing:
    bundle_candidates = [
        pathlib.Path(os.environ.get("CODEPAUSE_PHASE35_BUNDLE", "/content/phase3_5_bundle.zip")),
        root_path / "phase3_5_bundle.zip",
        pathlib.Path("/content/drive/MyDrive/codepause_phase3_5/phase3_5_bundle.zip"),
    ]
    bundle = next((p for p in bundle_candidates if p.exists()), None)
    if bundle:
        print(f"Overlaying Phase 3.5 bundle: {bundle}")
        with zipfile.ZipFile(bundle) as zf:
            zf.extractall(PROJECT_DIR)
        missing = missing_required()

if missing:
    raise RuntimeError(
        "Repo checkout is missing Phase 3.5 files: " + ", ".join(missing) + ". "
        "Fix by either pushing Phase 3.5 changes to GitHub or uploading phase3_5_bundle.zip to /content or Drive."
    )

print("Repo setup OK.")

In [ ]:
# 2. Runtime check
import torch

if not torch.cuda.is_available():
    print("CRITICAL: CUDA unavailable. Colab T4 runtime required.")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
print(f"GPU: {gpu_name}")

## 3. Tests

In [ ]:
!python -m pytest tests/test_phase3_prompt_template.py tests/test_phase3_evaluator_prompt_support.py tests/test_phase3_dataset_v3.py -v

## 4. Dataset v3 Validation

In [ ]:
!python eval/validate_dataset.py data/thinkanywhere_sft_v3.jsonl --schema sft
!python eval/validate_dataset.py data/thinkanywhere_sft_v3.jsonl --schema sft --min_examples 60

## 5. Base + Best Prompt Evaluation

In [ ]:
!python eval/evaluate_baseline.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --problems_path data/heldout_problems_30.jsonl \
  --output_path results/phase3_5_base_best_prompt.jsonl \
  --prompt_template best_phase3_prompt

## 5b. Training/Evaluation Dependencies

In [ ]:
# Install/check dependencies required by training and PEFT adapter loading
import importlib.metadata as md
import importlib.util
import subprocess
import sys

required = {
    "trl": "trl",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "transformers": "transformers",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

try:
    torchao_version = md.version("torchao")
except md.PackageNotFoundError:
    torchao_version = "0"
if tuple(int(part) for part in torchao_version.split(".")[:2] if part.isdigit()) < (0, 16):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torchao>=0.16.0"], check=True)

for pkg in ["torch", "peft", "torchao", "trl", "bitsandbytes"]:
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "not installed")

## 6. Adapter v3 Training

In [ ]:
!python training/sft_lora.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --dataset_path data/thinkanywhere_sft_v3.jsonl \
  --output_dir results/sft_phase3_5_v3 \
  --epochs 1 \
  --max_steps 150 \
  --max_seq_length 1024 \
  --device auto \
  --load_in_4bit \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --lora_r 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05

## 7. Adapter v3 + Best Prompt Evaluation

In [ ]:
!python eval/evaluate_finetuned.py \
  --base_model Qwen/Qwen1.5-1.8B-Chat \
  --adapter_path results/sft_phase3_5_v3 \
  --problems_path data/heldout_problems_30.jsonl \
  --output_path results/phase3_5_adapter_v3_best_prompt.jsonl \
  --prompt_template best_phase3_prompt

## 8. Report Generation

In [ ]:
!python eval/make_report.py \
  --baseline results/phase3_5_base_best_prompt.jsonl \
  --finetuned results/phase3_5_adapter_v3_best_prompt.jsonl \
  --out results/phase3_5_report.md

## 9. Artifact Copy to Drive

In [ ]:
import shutil, os

artifacts = [
    "results/phase3_5_base_best_prompt.jsonl",
    "results/phase3_5_adapter_v3_best_prompt.jsonl",
    "results/phase3_5_report.md",
    "results/phase3_5_dataset_v3_quality_report.md",
    "data/thinkanywhere_sft_v3.jsonl",
    "results/sft_phase3_5_v3",
]

for artifact in artifacts:
    dest = os.path.join(root_path, artifact)
    if os.path.exists(artifact):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(artifact):
            shutil.copytree(artifact, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(artifact, dest)
        print(f"Copied: {artifact} -> {dest}")
    else:
        print(f"MISSING: {artifact}")

## 10. Final Hard Gate Check

In [ ]:
import json, pathlib

def score(path):
    rows = [json.loads(line) for line in pathlib.Path(path).read_text().splitlines() if line.strip()]
    return sum(1 for row in rows if row.get("passed")), len(rows)

base_pass, base_total = score("results/phase3_5_base_best_prompt.jsonl")
adapter_pass, adapter_total = score("results/phase3_5_adapter_v3_best_prompt.jsonl")
print(f"base + best prompt: {base_pass}/{base_total}")
print(f"adapter v3 + best prompt: {adapter_pass}/{adapter_total}")
print(f"delta: {adapter_pass - base_pass}/{base_total}")
print("HARD_GATE:", "PASS" if adapter_pass > base_pass else "FAIL")